In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.datasets.mos2_sr import BTOSRDataset
from src.models.our_method.swin_cafm import SwinCAFM
from src.util.metrics import RMSE_surface_roughness_l1
from src.util.material_property_eval import calculate_roughness

### Load Model + Dataset

In [ ]:
MODEL_FP        = "__exps__/bto/2025-05-07_15-50-12_bto-4x-no-augs-sr-loss=rot-invariant-downsample=linear-model=SparseCAFM/bto-4x-no-augs-sr-loss=rot-invariant-downsample=linear-model=SparseCAFM_latest.pth"
MODEL_FP        = "__exps__/bto/2025-05-07_15-48-43_bto-4x-no-augs-sr-loss=flip-invariant-downsample=linear-model=SparseCAFM/bto-4x-no-augs-sr-loss=flip-invariant-downsample=linear-model=SparseCAFM_latest.pth"
MODEL_FP        = "__exps__/bto/2025-05-08_15-13-33_bto-4x-no-augs-sr-loss=sr-baseline=linear-model=SparseCAFM/bto-4x-no-augs-sr-loss=sr-baseline=linear-model=SparseCAFM_latest.pth"
dataset         = BTOSRDataset(split="val", upsample_factor=4)
model: SwinCAFM = torch.load(MODEL_FP).cuda();

In [ ]:
FOUR_X_FP = "data/raw-data/3-12-25/A4 128_Z Height_Backward_009.npy"

# load [64, 64] validation sample
sparse_data = np.load(FOUR_X_FP)
sparse_data = torch.Tensor(sparse_data)
sparse_sr   = calculate_roughness((sparse_data))

### Predict

In [ ]:
# -> normalize to nn distribution: [0, 1]
X = sparse_data.clone()
X = (X - dataset.topo_maps_min) / (dataset.topo_maps_max - dataset.topo_maps_min)

# X_hat
pred = model(X.cuda().unsqueeze(0))

# -> scale original topology map distribution (nm)f
pred_orig = (
    pred * (dataset.topo_maps_max - dataset.topo_maps_min) + dataset.topo_maps_min
)
pred_orig_np = pred_orig.squeeze(0).detach().cpu().numpy()

# get surface roughness
predicted_sr = calculate_roughness(pred_orig_np)

In [ ]:
# get the original, full-resolution topology map
gt = np.load("data/raw-data/3-12-25/A4 512_Z Height_Backward_011.npy")

# get surface roughness
ground_truth_sr = calculate_roughness(gt)

In [ ]:
sparse_data.shape, pred_orig_np.shape, gt.shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

fig = plt.figure(figsize=(18, 5.76), dpi=600)
width_ratios = [15, 15, 15, 1]
gs = gridspec.GridSpec(1, 4, width_ratios=width_ratios, wspace=0.01)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])
cax = fig.add_subplot(gs[0, 3])

image_axes = [ax1, ax2, ax3]
tensors = [sparse_data, pred_orig_np, gt]
titles = ["Sparse Input", "Prediction", "Ground Truth"]

mock_roughness_rms = [
    sparse_sr["RMS roughness (Sq)"],
    predicted_sr["RMS roughness (Sq)"],
    ground_truth_sr["RMS roughness (Sq)"],
]
mock_roughness_mean = [
    sparse_sr["Mean roughness (Sa)"],
    predicted_sr["Mean roughness (Sa)"],
    ground_truth_sr["Mean roughness (Sa)"],
]

im = None

for i, (ax, tensor, title) in enumerate(zip(image_axes, tensors, titles)):
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad(color="lightgrey")

    current_im = ax.imshow(tensor, cmap=cmap, interpolation="nearest")

    if i == 0:                       # 0 == Sparse Input
        h, w = tensor.shape[-2], tensor.shape[-1]   # explicit & safe
        # one minor tick per pixel edge
        ax.set_xticks(np.arange(-0.5, w, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, h, 1), minor=True)

        ax.grid(which='minor',
                color='white',
                linewidth=0.4,  
                alpha=0.4,    # thin but visible
                zorder=3)            # be sure it's on top

        # hide tick *marks* but keep axis ON so grid survives
        ax.tick_params(which='both',
                    bottom=False, left=False,
                    labelbottom=False, labelleft=False)

        # hide spines instead of using axis('off')
        for spine in ax.spines.values():
            spine.set_visible(False)
    else:
        # for the other panels we can still turn the axis off
        ax.axis('off')

    if i == 2:  # keep handle for the global colour‑bar
        im = current_im
    ax.set_title(title, fontsize=14)

    # ---------- roughness text -----------
    rms_str = (
        f"{mock_roughness_rms[i]:.3f}" if not np.isnan(mock_roughness_rms[i]) else "N/A"
    )
    mean_str = (
        f"{mock_roughness_mean[i]:.3f}"
        if not np.isnan(mock_roughness_mean[i])
        else "N/A"
    )
    ax.text(
        0.03,
        0.97,
        f"RMS (Sq): {rms_str} nm\nMean (Sa): {mean_str} nm",
        transform=ax.transAxes,
        fontsize=10,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.75),
    )

    # ---------- add scale bar ONLY to ground‑truth panel -----------
    if i == 2:
        # ‑‑ choose a physical length you want to show (here 200 nm) and
        #    convert it to *data* units = pixels.
        wanted_nm = 200  # ← change to match your need
        nm_per_pixel = 2000 / 512  # ← calibration from your experiment
        bar_len_pixels = wanted_nm / nm_per_pixel

        scalebar = AnchoredSizeBar(
            ax.transData,
            bar_len_pixels,  # bar length in data units
            f"{wanted_nm} nm",  # text label
            loc="lower right",  # bottom‑right corner
            pad=0.4,  # offset from frame
            color="white",
            frameon=False,
            size_vertical=bar_len_pixels * 0.10,
            fontproperties=fm.FontProperties(size=9),
        )
        ax.add_artist(scalebar)

# ---------- global colour‑bar -----------
if im is not None:
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label("Surface Height (nm)", rotation=270, labelpad=15, fontsize=14)
else:
    cax.axis("off")

plt.show()

### Generate Figure